In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

all_data = pd.read_csv('../data/processed_combined_final.csv')
print(all_data.shape)

(2018473, 81)


In [ ]:
X = all_data.drop(columns=['Label', 'Label_Binary', 'Label_Multiclass'])
y_binary = all_data['Label_Binary']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_binary, test_size=0.2, random_state=42, stratify=y_binary
)

test_indices = X_test.index
y_test_multiclass = all_data.loc[test_indices, 'Label_Multiclass']

In [ ]:
from xgboost import XGBClassifier
import time

start = time.time()

xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    n_jobs=-1,
    eval_metric='logloss'
)
xgb_model.fit(X_train, y_train)

print(f"Training time: {time.time() - start:.2f} seconds")

Training time: 54.68 seconds


In [ ]:
y_pred_xgb = xgb_model.predict(X_test)

print(classification_report(y_test, y_pred_xgb, target_names=['BENIGN', 'ATTACK']))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_xgb))

              precision    recall  f1-score   support

      BENIGN       1.00      1.00      1.00    318547
      ATTACK       1.00      1.00      1.00     85148

    accuracy                           1.00    403695
   macro avg       1.00      1.00      1.00    403695
weighted avg       1.00      1.00      1.00    403695


Confusion Matrix:
[[318221    326]
 [   189  84959]]


In [ ]:
results_df_xgb = pd.DataFrame({
    'true_multiclass': y_test_multiclass,
    'predicted_binary': y_pred_xgb
})

attack_detection_rate_xgb = results_df_xgb[results_df_xgb['true_multiclass'] != 'BENIGN'].groupby('true_multiclass')['predicted_binary'].mean()
print(attack_detection_rate_xgb)

true_multiclass
Bot                         0.687042
DDoS                        0.999608
DoS GoldenEye               0.999036
DoS Hulk                    0.999740
DoS Slowhttptest            0.994094
DoS slowloris               0.999044
FTP-Patator                 0.995745
Other                       0.736842
PortScan                    0.999890
SSH-Patator                 0.996689
Web Attack - Brute Force    0.958333
Web Attack - XSS            0.969325
Name: predicted_binary, dtype: float64


In [ ]:
from sklearn.ensemble import IsolationForest

benign_only = X_train[y_train == 0]

iso_forest = IsolationForest(
    n_estimators=100,
    contamination=0.05,
    random_state=42,
    n_jobs=-1
)
iso_forest.fit(benign_only)

print("Isolation Forest trained on", benign_only.shape[0], "benign samples")

Isolation Forest trained on 1274185 benign samples


In [ ]:
iso_predictions = iso_forest.predict(X_test)

# Isolation Forest outputs 1 for "normal" and -1 for "anomaly" - convert to match our 0/1 format
iso_predictions_binary = np.where(iso_predictions == -1, 1, 0)

print(classification_report(y_test, iso_predictions_binary, target_names=['BENIGN', 'ATTACK']))

              precision    recall  f1-score   support

      BENIGN       0.85      0.95      0.90    318547
      ATTACK       0.68      0.40      0.50     85148

    accuracy                           0.83    403695
   macro avg       0.77      0.67      0.70    403695
weighted avg       0.82      0.83      0.82    403695



In [ ]:
results_df_iso = pd.DataFrame({
    'true_multiclass': y_test_multiclass,
    'predicted_binary': iso_predictions_binary
})

bot_detection_iso = results_df_iso[results_df_iso['true_multiclass'] != 'BENIGN'].groupby('true_multiclass')['predicted_binary'].mean()
print(bot_detection_iso)

true_multiclass
Bot                         0.024450
DDoS                        0.172330
DoS GoldenEye               0.094986
DoS Hulk                    0.817583
DoS Slowhttptest            0.228346
DoS slowloris               0.512428
FTP-Patator                 0.000000
Other                       0.789474
PortScan                    0.000330
SSH-Patator                 0.000000
Web Attack - Brute Force    0.050595
Web Attack - XSS            0.012270
Name: predicted_binary, dtype: float64


In [ ]:
from sklearn.ensemble import RandomForestClassifier
import time

start = time.time()

rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

print(f"Training time: {time.time() - start:.2f} seconds")

Training time: 833.54 seconds


In [ ]:
y_pred = rf_model.predict(X_test)
print(classification_report(y_test, y_pred, target_names=['BENIGN', 'ATTACK']))

              precision    recall  f1-score   support

      BENIGN       1.00      1.00      1.00    318547
      ATTACK       1.00      1.00      1.00     85148

    accuracy                           1.00    403695
   macro avg       1.00      1.00      1.00    403695
weighted avg       1.00      1.00      1.00    403695



In [ ]:
import joblib

joblib.dump(rf_model, '../models/random_forest_binary.pkl')
print("Model saved successfully")

Model saved successfully


In [ ]:
import mlflow
from pathlib import Path

mlflow_dir = Path("../mlruns").resolve()
mlflow.set_tracking_uri(mlflow_dir.as_uri())

print("MLflow tracking URI set to:", mlflow.get_tracking_uri())

MLflow tracking URI set to: file:///C:/Users/Malik%20Faizan%20Ali/Documents/network-intrusion-mlops/mlruns


In [ ]:
import mlflow

mlflow.set_tracking_uri("sqlite:///../mlflow.db")

print("MLflow tracking URI set to:", mlflow.get_tracking_uri())

MLflow tracking URI set to: sqlite:///../mlflow.db


In [ ]:
mlflow.set_experiment("network-intrusion-detection")

with mlflow.start_run(run_name="random_forest_binary_v1"):
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("model_type", "RandomForestClassifier")
    
    mlflow.log_metric("test_accuracy", (y_pred == y_test).mean())
    mlflow.log_metric("bot_detection_rate", 0.743276)
    
    mlflow.sklearn.log_model(rf_model, "model")
    
print("Logged to MLflow")

2026/07/05 15:49:35 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/07/05 15:49:35 INFO mlflow.store.db.utils: Updating database tables
2026/07/05 15:49:40 INFO mlflow.tracking.fluent: Experiment with name 'network-intrusion-detection' does not exist. Creating a new experiment.
2026/07/05 15:49:42 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Logged to MLflow


In [ ]:
import os
print(os.listdir('../models'))

['random_forest_binary.pkl']


In [ ]:
sample = X_test.iloc[0].to_dict()
print(sample)

{'Destination Port': 53.0, 'Flow Duration': 147.0, 'Total Fwd Packets': 2.0, 'Total Backward Packets': 2.0, 'Total Length of Fwd Packets': 58.0, 'Total Length of Bwd Packets': 90.0, 'Fwd Packet Length Max': 29.0, 'Fwd Packet Length Min': 29.0, 'Fwd Packet Length Mean': 29.0, 'Fwd Packet Length Std': 0.0, 'Bwd Packet Length Max': 45.0, 'Bwd Packet Length Min': 45.0, 'Bwd Packet Length Mean': 45.0, 'Bwd Packet Length Std': 0.0, 'Flow Bytes/s': 1006802.721, 'Flow Packets/s': 27210.88435, 'Flow IAT Mean': 49.0, 'Flow IAT Std': 79.67433715, 'Flow IAT Max': 141.0, 'Flow IAT Min': 3.0, 'Fwd IAT Total': 3.0, 'Fwd IAT Mean': 3.0, 'Fwd IAT Std': 0.0, 'Fwd IAT Max': 3.0, 'Fwd IAT Min': 3.0, 'Bwd IAT Total': 3.0, 'Bwd IAT Mean': 3.0, 'Bwd IAT Std': 0.0, 'Bwd IAT Max': 3.0, 'Bwd IAT Min': 3.0, 'Fwd PSH Flags': 0.0, 'Bwd PSH Flags': 0.0, 'Fwd URG Flags': 0.0, 'Bwd URG Flags': 0.0, 'Fwd Header Length': 40.0, 'Bwd Header Length': 40.0, 'Fwd Packets/s': 13605.44218, 'Bwd Packets/s': 13605.44218, 'Min P

In [ ]:
print(y_test.iloc[0])

0


In [ ]:
print("True label:", y_test.iloc[0])

True label: 0


In [ ]:
import joblib

rf_model = joblib.load('../models/random_forest_binary.pkl')
print("Model loaded")

Model loaded


In [2]:
import mlflow
import mlflow.sklearn

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("network-intrusion-detection")

print("Tracking URI set to:", mlflow.get_tracking_uri())

c:\Users\Malik Faizan Ali\Documents\network-intrusion-mlops\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Tracking URI set to: http://localhost:5000


In [3]:
import joblib

rf_model = joblib.load('../models/random_forest_binary.pkl')

with mlflow.start_run(run_name="random_forest_binary_v2") as run:
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("model_type", "RandomForestClassifier")
    mlflow.log_metric("bot_detection_rate", 0.743276)
    
    mlflow.sklearn.log_model(
        rf_model, 
        "model",
        registered_model_name="network-intrusion-classifier"
    )

print("Random Forest registered successfully")

2026/07/08 12:04:10 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Registered model 'network-intrusion-classifier' already exists. Creating a new version of this model...
2026/07/08 12:05:27 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: network-intrusion-classifier, version 2
Created version '2' of model 'network-intrusion-classifier'.


🏃 View run random_forest_binary_v2 at: http://localhost:5000/#/experiments/1/runs/d83cc7c5df444f80b8c1446cd16c3d69
🧪 View experiment at: http://localhost:5000/#/experiments/1
Random Forest registered successfully


In [8]:
import os
print(os.path.exists('../data/processed_combined_final.csv'))

True


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

all_data = pd.read_csv('../data/processed_combined_final.csv')
X = all_data.drop(columns=['Label', 'Label_Binary', 'Label_Multiclass'])
y_binary = all_data['Label_Binary']

X_train, X_test, y_train, y_test = train_test_split(
    X, y_binary, test_size=0.2, random_state=42, stratify=y_binary
)

print("X_train shape:", X_train.shape)

X_train shape: (1614778, 78)


In [3]:
from sklearn.ensemble import IsolationForest

benign_only = X_train[y_train == 0]

iso_forest = IsolationForest(
    n_estimators=100,
    contamination=0.05,
    random_state=42,
    n_jobs=-1
)
iso_forest.fit(benign_only)

print("Isolation Forest trained on", benign_only.shape[0], "benign samples")

Isolation Forest trained on 1274185 benign samples


In [ ]:
import mlflow
import mlflow.sklearn

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("network-intrusion-detection")

with mlflow.start_run(run_name="isolation_forest_v1") as run:
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("contamination", 0.05)
    mlflow.log_param("model_type", "IsolationForest")
    
    mlflow.sklearn.log_model(
        iso_forest,
        "model",
        registered_model_name="network-intrusion-anomaly-detector"
    )

print("Isolation Forest registered successfully")

c:\Users\Malik Faizan Ali\Documents\network-intrusion-mlops\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026/07/12 18:18:34 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
